## Example 1: Basic Spatial Autocorrelation Analysis

In [ ]:
# Example 1: Basic Spatial Autocorrelation Analysis

import esda
import numpy as np
from esda import Moran
from libpysal.weights import full2W, lat2W
import warnings
warnings.filterwarnings('ignore', category=FutureWarning)

# Demonstrate basic spatial autocorrelation analysis
print("=== Basic Spatial Autocorrelation Analysis ===")

try:
    # Create sample spatial data
    np.random.seed(42)  # For reproducible results
    
    print("=== Sample Data Generation ===")
    
    # Generate sample spatial values (e.g., income levels across regions)
    n_regions = 25  # 5x5 grid of regions
    spatial_values = np.random.normal(50000, 15000, n_regions)  # Income data
    
    # Create a simple spatial weights matrix (rook contiguity for 5x5 grid)
    def create_rook_weights_matrix(rows, cols):
        """Create rook contiguity weights matrix for a grid."""
        n = rows * cols
        weights = np.zeros((n, n))
        
        for i in range(rows):
            for j in range(cols):
                current = i * cols + j
                
                # Add neighbors (rook contiguity - only adjacent cells)
                neighbors = []
                if i > 0:  # North neighbor
                    neighbors.append((i-1) * cols + j)
                if i < rows-1:  # South neighbor
                    neighbors.append((i+1) * cols + j)
                if j > 0:  # West neighbor
                    neighbors.append(i * cols + (j-1))
                if j < cols-1:  # East neighbor
                    neighbors.append(i * cols + (j+1))
                
                # Set weights (row-standardized)
                if neighbors:
                    weight = 1.0 / len(neighbors)
                    for neighbor in neighbors:
                        weights[current, neighbor] = weight
        
        return weights
    
    # Create 5x5 grid weights matrix
    weights_matrix = create_rook_weights_matrix(5, 5)
    
    print(f"Generated spatial data:")
    print(f"  Number of regions: {n_regions}")
    print(f"  Value range: ${spatial_values.min():.0f} - ${spatial_values.max():.0f}")
    print(f"  Mean value: ${spatial_values.mean():.0f}")
    print(f"  Weights matrix shape: {weights_matrix.shape}")
    print(f"  Non-zero weights: {np.count_nonzero(weights_matrix)}")
    
    # Convert weights matrix to the libpysal W that esda takes
    weights_obj = full2W(weights_matrix)
    rook = lat2W(5, 5)  # libpysal's own rook lattice must agree with the hand-built matrix
    assert weights_obj.n == n_regions and {k: set(v) for k, v in weights_obj.neighbors.items()} == {k: set(v) for k, v in rook.neighbors.items()}
    
    # Compute Moran's I statistic
    print(f"\n=== Moran's I Computation ===")
    
    # Calculate Moran's I for spatial autocorrelation
    moran_result = Moran(spatial_values, weights_obj)
    
    print(f"Moran's I Analysis Results:")
    print(f"  Moran's I statistic: {moran_result.I:.4f}")
    print(f"  Expected I (under null): {moran_result.EI:.4f}")
    print(f"  Variance of I: {moran_result.VI_norm:.6f}")
    print(f"  Z-score: {moran_result.z_norm:.4f}")
    print(f"  P-value (two-tailed): {moran_result.p_norm:.4f}")
    assert np.isclose(moran_result.EI, -1.0 / (n_regions - 1)) and 0.0 <= moran_result.p_norm <= 1.0
    
    # Interpret results
    print(f"\n=== Interpretation ===")
    
    if moran_result.I > moran_result.EI:
        if moran_result.p_norm < 0.05:
            interpretation = "Significant positive spatial autocorrelation"
            meaning = "Similar values tend to cluster together"
        else:
            interpretation = "Weak positive spatial autocorrelation (not significant)"
            meaning = "Slight tendency for similar values to cluster"
    elif moran_result.I < moran_result.EI:
        if moran_result.p_norm < 0.05:
            interpretation = "Significant negative spatial autocorrelation"
            meaning = "Dissimilar values tend to be neighbors"
        else:
            interpretation = "Weak negative spatial autocorrelation (not significant)"
            meaning = "Slight tendency for dissimilar values to be neighbors"
    else:
        interpretation = "No spatial autocorrelation"
        meaning = "Random spatial pattern"
    
    print(f"  Result: {interpretation}")
    print(f"  Meaning: {meaning}")
    print(f"  Significance level: {'Significant' if moran_result.p_norm < 0.05 else 'Not significant'} at α=0.05")
    
    # Display sample spatial pattern
    print(f"\n=== Spatial Pattern Visualization (5x5 Grid) ===")
    grid_values = spatial_values.reshape(5, 5)
    
    print("Sample income values (in thousands):")
    for i in range(5):
        row_str = " ".join([f"{val/1000:5.1f}" for val in grid_values[i]])
        print(f"  {row_str}")
    
    print(f"\nBasic spatial autocorrelation analysis completed successfully")
    
except Exception as e:
    print(f"Spatial autocorrelation analysis completed with note: {e}")
    raise AssertionError(
        f"esda example (cell 1) failed: {type(e).__name__}: {e}"
    ) from e

## Example 2: Local Indicators of Spatial Association (LISA)

In [ ]:
# Example 2: Local Indicators of Spatial Association (LISA)

import esda
import numpy as np
from esda import Moran_Local
from libpysal.weights import W, lat2W
import warnings
warnings.filterwarnings('ignore')

# Demonstrate Local Indicators of Spatial Association (LISA) analysis
print("=== Local Indicators of Spatial Association (LISA) ===")

try:
    # Generate sample data with spatial clustering pattern
    np.random.seed(123)
    
    print("=== Sample Data with Spatial Clustering ===")
    
    # Create 6x6 grid with intentional clustering
    n_regions = 36
    base_values = np.random.normal(100, 20, n_regions)
    
    # Add clustering: higher values in center, lower on edges
    grid_size = 6
    values = base_values.copy()
    
    for i in range(grid_size):
        for j in range(grid_size):
            idx = i * grid_size + j
            
            # Distance from center
            center_i, center_j = grid_size // 2, grid_size // 2
            dist_from_center = max(abs(i - center_i), abs(j - center_j))
            
            # Apply clustering effect
            if dist_from_center <= 1:  # Center cluster
                values[idx] += 50  # Higher values
            elif dist_from_center >= 3:  # Edge regions
                values[idx] -= 30  # Lower values
    
    print(f"Generated clustered spatial data:")
    print(f"  Grid size: {grid_size}x{grid_size}")
    print(f"  Total regions: {n_regions}")
    print(f"  Value range: {values.min():.1f} - {values.max():.1f}")
    print(f"  Mean value: {values.mean():.1f}")
    
    # Create spatial weights matrix for 6x6 grid
    def create_queen_weights(rows, cols):
        """Create queen contiguity weights (includes diagonal neighbors)."""
        n = rows * cols
        weights_dict = {}
        
        for i in range(rows):
            for j in range(cols):
                current = i * cols + j
                neighbors = []
                
                # Add all 8 surrounding neighbors (queen contiguity)
                for di in [-1, 0, 1]:
                    for dj in [-1, 0, 1]:
                        if di == 0 and dj == 0:
                            continue
                        
                        ni, nj = i + di, j + dj
                        if 0 <= ni < rows and 0 <= nj < cols:
                            neighbor = ni * cols + nj
                            neighbors.append(neighbor)
                
                weights_dict[current] = neighbors
        
        return weights_dict
    
    weights_dict = create_queen_weights(grid_size, grid_size)
    
    # esda takes a libpysal W; build one from the neighbour dict (Moran_Local row-standardises it itself)
    weights = W(weights_dict)
    queen = lat2W(grid_size, grid_size, rook=False)  # libpysal's own queen lattice must agree
    assert weights.n == n_regions and {k: set(v) for k, v in weights.neighbors.items()} == {k: set(v) for k, v in queen.neighbors.items()}
    
    # Perform LISA analysis
    print(f"\n=== Local Moran's I Analysis ===")
    
    # Calculate Local Moran's I for each region
    lisa_result = Moran_Local(values, weights, seed=123)
    assert len(lisa_result.Is) == n_regions and ((0 < lisa_result.p_sim) & (lisa_result.p_sim <= 1)).all()
    
    print(f"LISA Analysis Results:")
    print(f"  Number of regions analyzed: {len(lisa_result.Is)}")
    print(f"  Local I statistics range: {lisa_result.Is.min():.4f} to {lisa_result.Is.max():.4f}")
    
    # Count significant local associations
    significant_count = sum(1 for p in lisa_result.p_sim if p < 0.05)
    print(f"  Significant local associations (p<0.05): {significant_count}")
    
    # Classify spatial association types
    high_high = sum(1 for i, (val, p) in enumerate(zip(lisa_result.Is, lisa_result.p_sim)) 
                   if val > 0 and p < 0.05 and values[i] > values.mean())
    low_low = sum(1 for i, (val, p) in enumerate(zip(lisa_result.Is, lisa_result.p_sim)) 
                 if val > 0 and p < 0.05 and values[i] < values.mean())
    high_low = sum(1 for i, (val, p) in enumerate(zip(lisa_result.Is, lisa_result.p_sim)) 
                  if val < 0 and p < 0.05 and values[i] > values.mean())
    low_high = sum(1 for i, (val, p) in enumerate(zip(lisa_result.Is, lisa_result.p_sim)) 
                  if val < 0 and p < 0.05 and values[i] < values.mean())
    
    print(f"\n=== Spatial Association Types ===")
    print(f"  High-High clusters: {high_high} (high values near high values)")
    print(f"  Low-Low clusters: {low_low} (low values near low values)")
    print(f"  High-Low outliers: {high_low} (high values near low values)")
    print(f"  Low-High outliers: {low_high} (low values near high values)")
    assert high_high > 0, "the +50 centre block was built as a cluster and must read High-High"
    
    # Display spatial pattern with LISA results
    print(f"\n=== Spatial Pattern with LISA Results ===")
    
    grid_display = np.zeros((grid_size, grid_size), dtype='<U10')
    
    for i in range(grid_size):
        for j in range(grid_size):
            idx = i * grid_size + j
            value = values[idx]
            local_i = lisa_result.Is[idx]
            p_value = lisa_result.p_sim[idx]
            
            if p_value < 0.05:  # Significant
                if local_i > 0 and value > values.mean():
                    pattern = "HH"  # High-High
                elif local_i > 0 and value < values.mean():
                    pattern = "LL"  # Low-Low
                elif local_i < 0 and value > values.mean():
                    pattern = "HL"  # High-Low
                else:
                    pattern = "LH"  # Low-High
            else:
                pattern = "NS"  # Not Significant
            
            grid_display[i, j] = f"{value:5.0f}{pattern}"
    
    print("Spatial pattern (Value + Association Type):")
    print("  Legend: HH=High-High, LL=Low-Low, HL=High-Low, LH=Low-High, NS=Not Significant")
    for row in grid_display:
        print("  " + "  ".join(row))
    
    # Summary statistics
    print(f"\n=== LISA Summary Statistics ===")
    positive_autocorr = sum(1 for val in lisa_result.Is if val > 0)
    negative_autocorr = sum(1 for val in lisa_result.Is if val < 0)
    
    print(f"  Regions with positive local autocorrelation: {positive_autocorr}")
    print(f"  Regions with negative local autocorrelation: {negative_autocorr}")
    print(f"  Most clustered region (highest Local I): {lisa_result.Is.argmax()} (I={lisa_result.Is.max():.4f})")
    print(f"  Most isolated region (lowest Local I): {lisa_result.Is.argmin()} (I={lisa_result.Is.min():.4f})")
    
    print("Local spatial association analysis completed successfully")
    
except Exception as e:
    print(f"LISA analysis completed with note: {e}")
    raise AssertionError(
        f"esda example (cell 3) failed: {type(e).__name__}: {e}"
    ) from e

## Example 3: Join Count Statistics for Categorical Data

In [ ]:
# Example 3: Join Count Statistics for Categorical Data

import esda
import numpy as np
from esda import Join_Counts
from libpysal.weights import full2W
import warnings
warnings.filterwarnings('ignore')

# Demonstrate join count statistics for categorical spatial data
print("=== Join Count Statistics for Categorical Data ===")

try:
    # Generate sample categorical data
    np.random.seed(456)
    
    print("=== Sample Categorical Data Generation ===")
    
    # Create 5x5 grid with land use categories
    grid_size = 5
    n_regions = grid_size * grid_size
    
    # Define land use categories
    categories = {
        0: "Residential", 
        1: "Commercial", 
        2: "Industrial", 
        3: "Agricultural",
        4: "Forest"
    }
    
    # Generate land use pattern with some clustering
    land_use = np.random.choice(list(categories.keys()), size=n_regions, 
                               p=[0.4, 0.2, 0.15, 0.15, 0.1])  # Weighted probabilities
    
    # Add some clustering by modifying adjacent cells
    for i in range(grid_size):
        for j in range(grid_size):
            current_idx = i * grid_size + j
            
            # Create some commercial clusters
            if i <= 1 and j <= 1:  # Top-left corner
                land_use[current_idx] = 1  # Commercial
            
            # Create some forest clusters  
            elif i >= 3 and j >= 3:  # Bottom-right corner
                land_use[current_idx] = 4  # Forest
    
    print(f"Generated categorical spatial data:")
    print(f"  Grid size: {grid_size}x{grid_size}")
    print(f"  Total regions: {n_regions}")
    print(f"  Categories: {len(categories)}")
    
    # Count categories
    category_counts = {}
    for cat_id, cat_name in categories.items():
        count = np.sum(land_use == cat_id)
        category_counts[cat_name] = count
        print(f"    {cat_name}: {count} regions")
    
    # Create adjacency matrix for join count analysis
    def create_adjacency_matrix(rows, cols):
        """Create binary adjacency matrix for rook contiguity."""
        n = rows * cols
        adjacency = np.zeros((n, n), dtype=int)
        
        for i in range(rows):
            for j in range(cols):
                current = i * cols + j
                
                # Add rook neighbors
                neighbors = []
                if i > 0:  # North
                    neighbors.append((i-1) * cols + j)
                if i < rows-1:  # South
                    neighbors.append((i+1) * cols + j)
                if j > 0:  # West
                    neighbors.append(i * cols + (j-1))
                if j < cols-1:  # East
                    neighbors.append(i * cols + (j+1))
                
                # Set adjacency
                for neighbor in neighbors:
                    adjacency[current, neighbor] = 1
        
        return adjacency
    
    adjacency_matrix = create_adjacency_matrix(grid_size, grid_size)
    
    print(f"  Adjacency matrix: {adjacency_matrix.shape}")
    print(f"  Total adjacencies: {np.sum(adjacency_matrix) // 2}")  # Divide by 2 for undirected
    
    # Display spatial pattern
    print(f"\n=== Spatial Land Use Pattern ===")
    
    land_use_grid = land_use.reshape(grid_size, grid_size)
    
    print("Land use pattern (by category ID):")
    for i in range(grid_size):
        row_str = " ".join([f"{land_use_grid[i,j]}" for j in range(grid_size)])
        print(f"  {row_str}")
    
    print("\nLand use pattern (by category name):")
    for i in range(grid_size):
        row_names = [categories[land_use_grid[i,j]][:4] for j in range(grid_size)]  # First 4 chars
        row_str = " ".join([f"{name:4s}" for name in row_names])
        print(f"  {row_str}")
    
    # Perform join count analysis
    print(f"\n=== Join Count Analysis ===")
    
    # Create binary variables for each category
    join_count_results = {}
    
    for cat_id, cat_name in categories.items():
        if np.sum(land_use == cat_id) > 0:  # Only analyze categories that exist
            # Create binary variable (1 if category, 0 otherwise)
            binary_var = (land_use == cat_id).astype(int)
            
            # Perform join count analysis
            jc_result = Join_Counts(binary_var, full2W(adjacency_matrix))
            assert jc_result.bb + jc_result.bw + jc_result.ww == jc_result.J == np.sum(adjacency_matrix) // 2, (jc_result.bb, jc_result.bw, jc_result.ww, jc_result.J)
            
            join_count_results[cat_name] = {
                'bb': jc_result.bb,  # Both neighbors are 1 (same category)
                'ww': jc_result.ww,  # Both neighbors are 0 (different categories)  
                'bw': jc_result.bw,  # One neighbor is 1, other is 0
                'J': jc_result.J     # Total joins
            }
            
            print(f"\n{cat_name} Join Counts:")
            print(f"  Same-category joins (BB): {jc_result.bb}")
            print(f"  Different-category joins (BW): {jc_result.bw}")
            print(f"  Non-category joins (WW): {jc_result.ww}")
            print(f"  Total joins: {jc_result.J}")
            
            # Calculate clustering tendency
            total_category_joins = jc_result.bb + jc_result.bw
            if total_category_joins > 0:
                clustering_ratio = jc_result.bb / total_category_joins
                print(f"  Clustering ratio: {clustering_ratio:.3f}")
                
                if clustering_ratio > 0.5:
                    print(f"  Interpretation: Tends to cluster")
                else:
                    print(f"  Interpretation: Tends to disperse")
    
    # Overall spatial pattern analysis
    print(f"\n=== Overall Spatial Pattern Analysis ===")
    
    # Calculate total same-category adjacent pairs
    total_same_adjacent = 0
    total_different_adjacent = 0
    
    for i in range(grid_size):
        for j in range(grid_size):
            current_idx = i * grid_size + j
            current_category = land_use[current_idx]
            
            # Check right neighbor
            if j < grid_size - 1:
                right_idx = i * grid_size + (j + 1)
                right_category = land_use[right_idx]
                
                if current_category == right_category:
                    total_same_adjacent += 1
                else:
                    total_different_adjacent += 1
            
            # Check bottom neighbor
            if i < grid_size - 1:
                bottom_idx = (i + 1) * grid_size + j
                bottom_category = land_use[bottom_idx]
                
                if current_category == bottom_category:
                    total_same_adjacent += 1
                else:
                    total_different_adjacent += 1
    
    total_adjacent = total_same_adjacent + total_different_adjacent
    overall_clustering = total_same_adjacent / total_adjacent if total_adjacent > 0 else 0
    
    print(f"Overall pattern statistics:")
    print(f"  Total adjacent pairs: {total_adjacent}")
    print(f"  Same-category pairs: {total_same_adjacent}")
    print(f"  Different-category pairs: {total_different_adjacent}")
    print(f"  Overall clustering coefficient: {overall_clustering:.3f}")
    
    if overall_clustering > 0.5:
        print(f"  Pattern: Clustered (categories tend to group together)")
    elif overall_clustering < 0.3:
        print(f"  Pattern: Dispersed (categories are well-mixed)")
    else:
        print(f"  Pattern: Random (moderate mixing)")
    
    # Category diversity index
    shannon_diversity = -sum(p * np.log(p) for p in [count/n_regions for count in category_counts.values()] if p > 0)
    max_diversity = np.log(len([c for c in category_counts.values() if c > 0]))
    evenness = shannon_diversity / max_diversity if max_diversity > 0 else 0
    
    print(f"\nSpatial diversity:")
    print(f"  Shannon diversity index: {shannon_diversity:.3f}")
    print(f"  Evenness index: {evenness:.3f}")
    print(f"  Interpretation: {'High diversity' if evenness > 0.8 else 'Moderate diversity' if evenness > 0.5 else 'Low diversity'}")
    
    print("Join count statistics analysis completed successfully")
    
except Exception as e:
    print(f"Join count analysis completed with note: {e}")
    raise AssertionError(
        f"esda example (cell 5) failed: {type(e).__name__}: {e}"
    ) from e

## Example 4: Spatial Smoothing and Rate Stabilization

In [ ]:
# Example 4: Spatial Smoothing and Rate Stabilization

import esda
import numpy as np
from esda import smoothing
import warnings
warnings.filterwarnings('ignore')

# Demonstrate spatial smoothing techniques for rate stabilization
print("=== Spatial Smoothing and Rate Stabilization ===")

try:
    # Generate sample count data with population at risk
    np.random.seed(789)
    
    print("=== Sample Public Health Data Generation ===")
    
    # Create sample data representing disease cases and population
    n_regions = 30
    
    # Generate population data (population at risk)
    min_pop, max_pop = 5000, 50000
    population = np.random.randint(min_pop, max_pop, n_regions)
    
    # Generate disease cases with some regions having higher rates
    base_rate = 0.02  # 2% base disease rate
    
    # Create some regions with elevated risk
    risk_multipliers = np.ones(n_regions)
    high_risk_regions = np.random.choice(n_regions, 5, replace=False)  # 5 high-risk regions
    risk_multipliers[high_risk_regions] = np.random.uniform(2.0, 4.0, 5)  # 2-4x higher risk
    
    # Generate cases based on population and risk
    expected_cases = population * base_rate * risk_multipliers
    actual_cases = np.random.poisson(expected_cases)
    
    # Calculate crude rates
    crude_rates = (actual_cases / population) * 1000  # Per 1000 people
    
    print(f"Generated public health data:")
    print(f"  Number of regions: {n_regions}")
    print(f"  Population range: {population.min():,} - {population.max():,}")
    print(f"  Total population: {population.sum():,}")
    print(f"  Total cases: {actual_cases.sum()}")
    print(f"  Case range: {actual_cases.min()} - {actual_cases.max()}")
    print(f"  Crude rate range: {crude_rates.min():.2f} - {crude_rates.max():.2f} per 1000")
    print(f"  High-risk regions: {sorted(high_risk_regions)}")
    
    # Show variability in small vs large populations
    print(f"\n=== Rate Variability by Population Size ===")
    
    # Sort regions by population
    sorted_indices = np.argsort(population)
    small_pop_indices = sorted_indices[:5]  # 5 smallest populations
    large_pop_indices = sorted_indices[-5:]  # 5 largest populations
    
    print("Smallest populations:")
    for idx in small_pop_indices:
        print(f"  Region {idx:2d}: Pop={population[idx]:,}, Cases={actual_cases[idx]}, Rate={crude_rates[idx]:.2f}/1000")
    
    print("\nLargest populations:")
    for idx in large_pop_indices:
        print(f"  Region {idx:2d}: Pop={population[idx]:,}, Cases={actual_cases[idx]}, Rate={crude_rates[idx]:.2f}/1000")
    
    # Calculate coefficient of variation for small vs large populations
    small_pop_cv = np.std(crude_rates[small_pop_indices]) / np.mean(crude_rates[small_pop_indices])
    large_pop_cv = np.std(crude_rates[large_pop_indices]) / np.mean(crude_rates[large_pop_indices])
    
    print(f"\nRate variability:")
    print(f"  Small populations CV: {small_pop_cv:.3f}")
    print(f"  Large populations CV: {large_pop_cv:.3f}")
    print(f"  Small populations are {'more' if small_pop_cv > large_pop_cv else 'less'} variable")
    
    # Apply spatial smoothing techniques
    print(f"\n=== Spatial Smoothing Techniques ===")
    
    try:
        # 1. Empirical Bayes Smoothing
        print("1. Empirical Bayes Smoothing:")
        
        # Calculate overall rate for reference
        overall_rate = actual_cases.sum() / population.sum()
        
        # Simple Empirical Bayes adjustment
        # Shrinks individual rates toward overall rate based on population size
        smoothing_weights = population / (population + population.mean())
        eb_smoothed_rates = (smoothing_weights * crude_rates + 
                            (1 - smoothing_weights) * (overall_rate * 1000))
        
        print(f"  Overall rate: {overall_rate * 1000:.2f} per 1000")
        print(f"  Smoothed rate range: {eb_smoothed_rates.min():.2f} - {eb_smoothed_rates.max():.2f} per 1000")
        
        # Compare smoothing effect
        smoothing_effect = np.abs(crude_rates - eb_smoothed_rates)
        print(f"  Average smoothing effect: {smoothing_effect.mean():.2f}")
        print(f"  Max smoothing effect: {smoothing_effect.max():.2f}")
        
        # Show examples of smoothing
        print(f"  Smoothing examples:")
        for i in range(5):  # Show first 5 regions
            direction = "up" if eb_smoothed_rates[i] > crude_rates[i] else "down"
            print(f"    Region {i}: {crude_rates[i]:.2f} → {eb_smoothed_rates[i]:.2f} ({direction})")
        
    except Exception as e:
        print(f"Empirical Bayes error: {e}")
        raise
    
    try:
        # 2. Spatial Rate Smoothing using simple neighborhood averaging
        print(f"\n2. Spatial Neighborhood Smoothing:")
        
        # Create simple neighborhood structure (circular neighbors)
        def create_spatial_neighbors(n_regions, n_neighbors=3):
            """Create simple circular neighborhood structure."""
            neighbors = {}
            for i in range(n_regions):
                # Simple circular neighbors
                neighbor_list = []
                for j in range(1, n_neighbors + 1):
                    left_neighbor = (i - j) % n_regions
                    right_neighbor = (i + j) % n_regions
                    neighbor_list.extend([left_neighbor, right_neighbor])
                neighbors[i] = list(set(neighbor_list))  # Remove duplicates
            return neighbors
        
        spatial_neighbors = create_spatial_neighbors(n_regions, 2)
        
        # Apply spatial smoothing
        spatially_smoothed_rates = np.zeros(n_regions)
        
        for i in range(n_regions):
            neighbors = spatial_neighbors[i]
            
            # Include region itself and its neighbors
            all_regions = [i] + neighbors
            neighbor_cases = actual_cases[all_regions]
            neighbor_population = population[all_regions]
            
            # Calculate neighborhood rate
            if neighbor_population.sum() > 0:
                neighborhood_rate = neighbor_cases.sum() / neighbor_population.sum()
                spatially_smoothed_rates[i] = neighborhood_rate * 1000
            else:
                spatially_smoothed_rates[i] = crude_rates[i]
        
        print(f"  Spatial smoothed rate range: {spatially_smoothed_rates.min():.2f} - {spatially_smoothed_rates.max():.2f} per 1000")
        
        # Compare spatial smoothing effect
        spatial_smoothing_effect = np.abs(crude_rates - spatially_smoothed_rates)
        print(f"  Average spatial smoothing effect: {spatial_smoothing_effect.mean():.2f}")
        
        print(f"  Spatial smoothing examples:")
        for i in range(5):  # Show first 5 regions
            neighbors_str = str(spatial_neighbors[i][:3])  # Show first 3 neighbors
            direction = "up" if spatially_smoothed_rates[i] > crude_rates[i] else "down"
            print(f"    Region {i} (neighbors {neighbors_str}): {crude_rates[i]:.2f} → {spatially_smoothed_rates[i]:.2f} ({direction})")
        
    except Exception as e:
        print(f"Spatial smoothing error: {e}")
        raise
    
    # 3. Rate Stabilization Analysis
    print(f"\n=== Rate Stabilization Analysis ===")
    
    # Compare variability before and after smoothing
    crude_variance = np.var(crude_rates)
    eb_variance = np.var(eb_smoothed_rates)
    spatial_variance = np.var(spatially_smoothed_rates)
    
    print(f"Rate variance comparison:")
    print(f"  Crude rates variance: {crude_variance:.4f}")
    print(f"  Empirical Bayes variance: {eb_variance:.4f}")
    print(f"  Spatial smoothed variance: {spatial_variance:.4f}")
    
    print(f"\nVariance reduction:")
    print(f"  Empirical Bayes: {((crude_variance - eb_variance) / crude_variance * 100):.1f}%")
    print(f"  Spatial smoothing: {((crude_variance - spatial_variance) / crude_variance * 100):.1f}%")
    
    # Identify regions with largest changes
    print(f"\n=== Regions with Largest Smoothing Effects ===")
    
    eb_changes = np.abs(crude_rates - eb_smoothed_rates)
    spatial_changes = np.abs(crude_rates - spatially_smoothed_rates)
    
    # Find regions most affected by each type of smoothing
    most_eb_affected = np.argsort(eb_changes)[-3:]  # Top 3
    most_spatial_affected = np.argsort(spatial_changes)[-3:]  # Top 3
    
    print("Most affected by Empirical Bayes smoothing:")
    for idx in most_eb_affected:
        print(f"  Region {idx}: Pop={population[idx]:,}, {crude_rates[idx]:.2f} → {eb_smoothed_rates[idx]:.2f}")
    
    print("\nMost affected by spatial smoothing:")
    for idx in most_spatial_affected:
        print(f"  Region {idx}: Pop={population[idx]:,}, {crude_rates[idx]:.2f} → {spatially_smoothed_rates[idx]:.2f}")
    
    # Summary of smoothing benefits
    print(f"\n=== Smoothing Benefits Summary ===")
    benefits = [
        "✓ Reduces random variation in small populations",
        "✓ Improves rate reliability and comparability", 
        "✓ Borrows strength from neighboring regions",
        "✓ Helps identify genuine spatial patterns",
        "✓ Stabilizes rates for mapping and analysis"
    ]
    
    for benefit in benefits:
        print(f"  {benefit}")

    print("Spatial smoothing and rate stabilization analysis completed successfully")

except Exception as e:
    print(f"Spatial smoothing analysis completed with note: {e}")
    raise AssertionError(
        f"esda example (cell 7) failed: {type(e).__name__}: {e}"
    ) from e

## Example 5: Comprehensive Spatial Analysis Workflow

In [ ]:
# Example 5: Comprehensive Spatial Analysis Workflow

import esda
import numpy as np
from esda import Moran, Moran_Local, Join_Counts
from libpysal.weights import W, lat2W, full2W
import warnings
warnings.filterwarnings('ignore')

# Demonstrate comprehensive spatial analysis workflow combining multiple ESDA techniques
print("=== Comprehensive Spatial Analysis Workflow ===")

try:
    # Generate comprehensive spatial dataset
    np.random.seed(999)
    
    print("=== Comprehensive Spatial Dataset ===")
    
    # Create 7x7 grid representing a metropolitan area
    grid_size = 7
    n_regions = grid_size * grid_size
    
    # Generate multiple spatial variables
    # 1. Housing prices (continuous)
    base_price = 300000  # Base housing price
    price_noise = np.random.normal(0, 50000, n_regions)
    
    # Add spatial structure to prices (center high, edges lower)
    housing_prices = np.zeros(n_regions)
    for i in range(grid_size):
        for j in range(grid_size):
            idx = i * grid_size + j
            
            # Distance from center creates price gradient
            center_i, center_j = grid_size // 2, grid_size // 2
            dist_from_center = np.sqrt((i - center_i)**2 + (j - center_j)**2)
            
            # Price decreases with distance from center
            price_effect = base_price * (1 - 0.1 * dist_from_center)
            housing_prices[idx] = price_effect + price_noise[idx]
    
    # 2. Population density (continuous)
    base_density = 5000  # People per sq km
    density_noise = np.random.normal(0, 1000, n_regions)
    
    # Population density follows different pattern (higher near transport corridors)
    population_density = np.zeros(n_regions)
    for i in range(grid_size):
        for j in range(grid_size):
            idx = i * grid_size + j
            
            # Higher density along main diagonal (transport corridor)
            corridor_effect = 1.0
            if abs(i - j) <= 1:  # Near main diagonal
                corridor_effect = 2.0
            elif i == grid_size // 2 or j == grid_size // 2:  # Central cross
                corridor_effect = 1.5
            
            population_density[idx] = base_density * corridor_effect + density_noise[idx]
    
    # 3. Land use categories (categorical)
    land_use_categories = {
        0: "Residential",
        1: "Commercial", 
        2: "Industrial",
        3: "Mixed_Use",
        4: "Green_Space"
    }
    
    # Create clustered land use pattern
    land_use = np.random.choice(list(land_use_categories.keys()), 
                               size=n_regions, p=[0.5, 0.2, 0.1, 0.15, 0.05])
    
    # Add clustering to land use
    for i in range(grid_size):
        for j in range(grid_size):
            idx = i * grid_size + j
            
            # Commercial district in center
            if 2 <= i <= 4 and 2 <= j <= 4:
                if np.random.random() < 0.6:
                    land_use[idx] = 1  # Commercial
            
            # Industrial zone in corners
            elif (i <= 1 and j <= 1) or (i >= 5 and j >= 5):
                if np.random.random() < 0.4:
                    land_use[idx] = 2  # Industrial
            
            # Green space along edges
            elif i == 0 or i == grid_size-1 or j == 0 or j == grid_size-1:
                if np.random.random() < 0.3:
                    land_use[idx] = 4  # Green space
    
    print(f"Generated comprehensive spatial data:")
    print(f"  Grid size: {grid_size}x{grid_size} ({n_regions} regions)")
    print(f"  Housing prices: ${housing_prices.min():.0f} - ${housing_prices.max():.0f}")
    print(f"  Population density: {population_density.min():.0f} - {population_density.max():.0f} per sq km")
    print(f"  Land use categories: {len(land_use_categories)}")
    
    # Create spatial weights for analysis
    def create_comprehensive_weights(rows, cols):
        """Create spatial weights matrix with multiple connectivity options."""
        n = rows * cols
        weights_dict = {}
        
        for i in range(rows):
            for j in range(cols):
                current = i * cols + j
                neighbors = []
                
                # Rook contiguity (4 neighbors)
                directions = [(-1, 0), (1, 0), (0, -1), (0, 1)]
                for di, dj in directions:
                    ni, nj = i + di, j + dj
                    if 0 <= ni < rows and 0 <= nj < cols:
                        neighbor = ni * cols + nj
                        neighbors.append(neighbor)
                
                weights_dict[current] = neighbors
        
        return weights_dict
    
    spatial_weights_dict = create_comprehensive_weights(grid_size, grid_size)
    
    # Step 1: Global Spatial Autocorrelation Analysis
    print(f"\n=== Step 1: Global Spatial Autocorrelation ===")
    
    # esda takes a libpysal W; build one from the neighbour dict (Moran row-standardises it itself)
    weights = W(spatial_weights_dict)
    rook = lat2W(grid_size, grid_size)  # libpysal's own rook lattice must agree with the hand-built one
    assert weights.n == n_regions and {k: set(v) for k, v in weights.neighbors.items()} == {k: set(v) for k, v in rook.neighbors.items()}

    # Analyze housing prices
    price_moran = Moran(housing_prices, weights)
    print(f"Housing Prices Spatial Autocorrelation:")
    print(f"  Moran's I: {price_moran.I:.4f}")
    print(f"  P-value: {price_moran.p_norm:.4f}")
    print(f"  Interpretation: {'Clustered' if price_moran.I > price_moran.EI and price_moran.p_norm < 0.05 else 'Random/Dispersed'}")
    
    # Analyze population density
    density_moran = Moran(population_density, weights)
    print(f"\nPopulation Density Spatial Autocorrelation:")
    print(f"  Moran's I: {density_moran.I:.4f}")
    print(f"  P-value: {density_moran.p_norm:.4f}")
    print(f"  Interpretation: {'Clustered' if density_moran.I > density_moran.EI and density_moran.p_norm < 0.05 else 'Random/Dispersed'}")
    # the price surface falls off from the centre by construction, so it must read as clustered
    assert price_moran.I > price_moran.EI and price_moran.p_norm < 0.05, (price_moran.I, price_moran.p_norm)
    assert 0.0 <= density_moran.p_norm <= 1.0
    
    # Step 2: Local Spatial Analysis
    print(f"\n=== Step 2: Local Spatial Analysis (LISA) ===")
    
    # Local analysis of housing prices
    price_lisa = Moran_Local(housing_prices, weights, seed=999)
    assert len(price_lisa.Is) == n_regions and ((0 < price_lisa.p_sim) & (price_lisa.p_sim <= 1)).all()
    
    # Identify significant local clusters
    significant_regions = np.where(price_lisa.p_sim < 0.05)[0]
    
    print(f"Housing Price Local Analysis:")
    print(f"  Significant local associations: {len(significant_regions)}")
    
    if len(significant_regions) > 0:
        # Classify significant regions
        high_price_threshold = np.median(housing_prices)
        
        hh_clusters = []  # High-High
        ll_clusters = []  # Low-Low  
        hl_outliers = []  # High-Low
        lh_outliers = []  # Low-High
        
        for region in significant_regions:
            local_i = price_lisa.Is[region]
            price = housing_prices[region]
            
            if local_i > 0:  # Positive autocorrelation
                if price > high_price_threshold:
                    hh_clusters.append(region)
                else:
                    ll_clusters.append(region)
            else:  # Negative autocorrelation
                if price > high_price_threshold:
                    hl_outliers.append(region)
                else:
                    lh_outliers.append(region)
        
        print(f"  High-High price clusters: {len(hh_clusters)} regions")
        print(f"  Low-Low price clusters: {len(ll_clusters)} regions") 
        print(f"  High-Low outliers: {len(hl_outliers)} regions")
        print(f"  Low-High outliers: {len(lh_outliers)} regions")
        
    # Step 3: Categorical Spatial Analysis
    print(f"\n=== Step 3: Categorical Spatial Analysis ===")
    
    # Create adjacency matrix for join counts
    adjacency_matrix = np.zeros((n_regions, n_regions), dtype=int)
    for region, neighbors in spatial_weights_dict.items():
        for neighbor in neighbors:
            adjacency_matrix[region, neighbor] = 1
    
    # Analyze land use clustering for each category
    for cat_id, cat_name in land_use_categories.items():
        if np.sum(land_use == cat_id) > 1:  # Need at least 2 regions
            binary_var = (land_use == cat_id).astype(int)
            jc_result = Join_Counts(binary_var, full2W(adjacency_matrix))
            assert jc_result.bb + jc_result.bw + jc_result.ww == jc_result.J, (jc_result.bb, jc_result.bw, jc_result.ww, jc_result.J)
            
            total_cat_adjacencies = jc_result.bb + jc_result.bw
            clustering_ratio = jc_result.bb / total_cat_adjacencies if total_cat_adjacencies > 0 else 0
            
            print(f"{cat_name}:")
            print(f"  Regions: {np.sum(land_use == cat_id)}")
            print(f"  Same-category joins: {jc_result.bb}")
            print(f"  Clustering ratio: {clustering_ratio:.3f}")
            print(f"  Pattern: {'Clustered' if clustering_ratio > 0.5 else 'Dispersed'}")
    
    # Step 4: Correlation Analysis
    print(f"\n=== Step 4: Spatial Correlation Analysis ===")
    
    # Analyze relationship between variables
    price_density_corr = np.corrcoef(housing_prices, population_density)[0, 1]
    
    print(f"Variable correlations:")
    print(f"  Housing price vs Population density: {price_density_corr:.3f}")
    
    # Test if correlation is spatially structured
    try:
        # Simple spatial correlation test
        correlation_values = []
        
        for region in range(n_regions):
            neighbors = spatial_weights_dict.get(region, [])
            if neighbors:
                region_price = housing_prices[region]
                region_density = population_density[region]
                
                neighbor_prices = housing_prices[neighbors]
                neighbor_densities = population_density[neighbors]
                
                # Local correlation
                if len(neighbors) > 1:
                    local_corr = np.corrcoef(neighbor_prices, neighbor_densities)[0, 1]
                    if not np.isnan(local_corr):
                        correlation_values.append(local_corr)
        
        if correlation_values:
            spatial_corr_variation = np.std(correlation_values)
            print(f"  Local correlation variation: {spatial_corr_variation:.3f}")
            print(f"  Spatial structure: {'Variable' if spatial_corr_variation > 0.2 else 'Uniform'}")
        
    except Exception as e:
        print(f"Spatial correlation error: {e}")
        raise
    
    # Step 5: Summary and Recommendations
    print(f"\n=== Step 5: Analysis Summary and Recommendations ===")
    
    summary_points = []
    
    # Add findings based on analysis results
    if 'price_moran' in locals():
        if price_moran.I > price_moran.EI and price_moran.p_norm < 0.05:
            summary_points.append("Housing prices show significant spatial clustering")
        
    if 'density_moran' in locals():
        if density_moran.I > density_moran.EI and density_moran.p_norm < 0.05:
            summary_points.append("Population density exhibits spatial autocorrelation")
    
    if abs(price_density_corr) > 0.3:
        relationship = "positive" if price_density_corr > 0 else "negative"
        summary_points.append(f"Strong {relationship} correlation between prices and density")
    
    summary_points.append("Land use shows mixed clustering patterns across categories")
    summary_points.append("Multiple spatial processes operating at different scales")
    
    print("Key findings:")
    for i, point in enumerate(summary_points, 1):
        print(f"  {i}. {point}")
    
    print(f"\nRecommended follow-up analyses:")
    recommendations = [
        "Investigate drivers of price clustering patterns",
        "Examine transportation accessibility effects", 
        "Analyze temporal changes in spatial patterns",
        "Consider spatial regression modeling",
        "Evaluate policy implications of spatial clusters"
    ]
    
    for i, rec in enumerate(recommendations, 1):
        print(f"  {i}. {rec}")

    print("Comprehensive spatial analysis workflow completed successfully")

except Exception as e:
    print(f"Comprehensive spatial analysis completed with note: {e}")
    raise AssertionError(
        f"esda example (cell 9) failed: {type(e).__name__}: {e}"
    ) from e